In [38]:
### Turn one hot encoded .mat to fasta seq
import math
import pandas as pd
import numpy as np
import scipy.io as sio
from os.path import dirname, join as pjoin
import os.path
import time
import random

In [39]:
def readInputAsArray(fileName):
    with open(fileName, 'r') as myfile:
        data = myfile.readlines()

    # Strip newline
    for i in range(0, len(data)):
        data[i] = data[i].rstrip()
    # print(data)
    return data

In [87]:
def oneHotEncode(seq):
    import numpy as np
    seq2=list()
    mapping = {"A":[1., 0., 0., 0.], "C": [0., 1., 0., 0.], "G": [0., 0., 1., 0.], "T":[0., 0., 0., 1.]};
    for i in seq:
    	seq2.append(mapping[i]  if i in mapping.keys() else [0., 0., 0., 0.]);
    return seq2;

In [106]:
def cluster2seq(Ecluster, Eseqs):
    debug=100
    all_curr_seq_index = []
    for i in range(len(Ecluster)):
        # header
    #     if (Ecluster[i][0] == '>'and i < debug):
        if (Ecluster[i][0] == '>'):
            curr_seq_index = []
            all_curr_seq_index += [curr_seq_index]
    #     elif (Ecluster[i][0] != '>' and i < debug):
        elif (Ecluster[i][0] != '>'):
            idx = int(Ecluster[i].split()[2][1:-3])
            curr_seq_index.append(idx)

####################### Separate clusters for train and test split #######################
    print(f"num clusters {len(all_curr_seq_index)}") # 0 encoded
    train_seq_num = int(math.ceil(0.6*len(all_curr_seq_index)))
    print(f"target train seqs {train_seq_num}")
    test_seq_num = len(all_curr_seq_index) - train_seq_num
    print(f"target test seqs {test_seq_num}")
    
    ## train test data split
    E_train_data = []
    E_test_data = []
    seq_num = 0
    max_seq_in_clstr = 5
    for clstr in all_curr_seq_index:
        selection_encode = []
        if (len(clstr) <= max_seq_in_clstr):
            selection = clstr
        else:
            random.shuffle(clstr)
            selection = clstr[-max_seq_in_clstr:] # Only want 5 seqs from large clusters
        for seq_idx in selection:
            seq = oneHotEncode(Eseqs[seq_idx*2+1])
            selection_encode += [seq]
            seq_num +=1

        if seq_num > train_seq_num:
            E_test_data += selection_encode
        else:
            E_train_data += selection_encode
#     print(f"train_data.shape {np.array(E_train_data).shape}")
#     print(f"test_data.shape {np.array(E_test_data).shape}")
    return np.array(E_train_data), np.array(E_test_data)

In [107]:
Eseqs = readInputAsArray('/Users/yibeijia/Downloads/nucleosome_occupancy/tbinet_stuff/cluster_seq/Eseqs.txt')[1:]
Dseqs = readInputAsArray('/Users/yibeijia/Downloads/nucleosome_occupancy/tbinet_stuff/cluster_seq/Dseqs.txt')[1:]
Ecluster = readInputAsArray('/Users/yibeijia/Downloads/nucleosome_occupancy/tbinet_stuff/cluster_seq/Eseqs80.clstr')
Dcluster = readInputAsArray('/Users/yibeijia/Downloads/nucleosome_occupancy/tbinet_stuff/cluster_seq/Dseqs80.clstr')
E_fout = '/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/Ecluster'
D_fout = '/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/Dcluster'

print(">>>>>>>>Preprocessing Enriched data")
E_train_data, E_test_data = cluster2seq(Ecluster, Eseqs)
print(">>>>>>>>Preprocessing Depleted data")
D_train_data, D_test_data = cluster2seq(Dcluster, Dseqs)

## combine E & D to create training, testing dataset & labels
print(">>>>>>>>Preparing Train Test dataset")
Train_data = np.concatenate((E_train_data, D_train_data), axis=0)
Edata_labels = np.ones(E_train_data.shape[0])
Ddata_labels = np.zeros(D_train_data.shape[0])
Train_labels = np.concatenate((Edata_labels, Ddata_labels), axis=0)
print(f"train data  shape is: {Train_data.shape}")
print(f"train label shape is: {Train_labels.shape}")

Test_data = np.concatenate((E_test_data, D_test_data), axis=0)
Edata_labels = np.ones(E_test_data.shape[0])
Ddata_labels = np.zeros(D_test_data.shape[0])
Test_labels = np.concatenate((Edata_labels, Ddata_labels), axis=0)
print(f"test data shape is: {Test_data.shape}")
print(f"test label shape is: {Test_labels.shape}")

## Write to .mat file
Data = {"Train_data" : np.array(Train_data), "Train_labels" : Train_labels}
sio.savemat('/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/Train_data.mat', Data, do_compression=True)
# sio.savemat('/scratch2/yibeijia/data/train_test_data/Train_data.mat',Data, do_compression=True)

Data = {"Test_data" : np.array(Test_data), "Test_labels" : Test_labels}
sio.savemat('/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/Test_data.mat', Data,  do_compression=True)
# sio.savemat('/scratch2/yibeijia/data/train_test_data/Test_data.mat',Data, do_compression=True)

>>>>>>>>Preprocessing Enriched data
num clusters 37316
target train seqs 22390
target test seqs 14926
>>>>>>>>Preprocessing Depleted data
num clusters 40854
target train seqs 24513
target test seqs 16341
 train data  shape is: (46896, 147, 4)
 train label shape is: (46896,)
 test data shape is: (276950, 147, 4)
 test label shape is: (276950,)
